# CASMI 2026: Fast timsTOF Spectra EDA + Dual-Engine Baseline

**Author**: Maman Salto

A self-contained starter guide and fast exploratory analysis for **Enveda - CASMI 2026: Molecule Identification from High-Resolution Mass Spectra**.
Covers:
1. Spectra inspection across ionization modes (`[M+H]+`, `[M-H]-`, `[M+Na]+`, `[M+CH2O2-H]-`).
2. Neutral mass calculation & candidate formula filtering.
3. Collision energy distribution analysis.
4. First-match MRR@25 evaluation metric mechanics & tautomer canonicalization (`InChIKey14`).


In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Loading CASMI 2026 test and train metadata...")
comp = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'
if not os.path.exists(comp):
    hits = glob.glob('/kaggle/input/**/test.parquet', recursive=True)
    comp = os.path.dirname(hits[0])

print("Data directory:", comp)
test_df = pd.read_parquet(os.path.join(comp, 'test.parquet'))
print(f"Test queries loaded: {len(test_df):,} spectra across {test_df.molecule_id.nunique():,} unique molecules.")


## 1. Ionization Mode & Adduct Distribution
In tandem mass spectrometry, precursor adduct determines the mass shift needed to find neutral mass:
- $[M+H]^+ \implies m_N = m_z - 1.007276$
- $[M-H]^- \implies m_N = m_z + 1.007276$
- $[M+Na]^+ \implies m_N = m_z - 22.989218$


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

test_df['adduct'].value_counts().plot(kind='bar', ax=axes[0], color='#2a78d6', edgecolor='black')
axes[0].set_title('Adduct Distribution in Test Queries', fontweight='bold')
axes[0].set_ylabel('Number of Spectra')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

test_df['ionization_mode'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#45aaf2', '#fc5c65'])
axes[1].set_title('Ionization Mode Share', fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()


## 2. Multi-Energy Spectra per Molecule
Unlike standard benchmarks with a single spectrum per analyte, timsTOF acquires stepped collision energies.


In [ ]:
spec_per_mol = test_df.groupby('molecule_id').size()
print(f"Mean spectra per molecule: {spec_per_mol.mean():.2f} (min {spec_per_mol.min()}, max {spec_per_mol.max()})")

fig, ax = plt.subplots(figsize=(8, 3.5))
spec_per_mol.plot(kind='hist', bins=range(1, 10), ax=ax, color='#26de81', edgecolor='black', rwidth=0.85)
ax.set_title("Distribution of Spectra per Molecule", fontweight='bold')
ax.set_xlabel("Spectra Count")
ax.set_ylabel("Molecules")
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 3. MS/MS Peak Counts and Fragmentation Richness


In [ ]:
test_df['n_peaks'] = test_df['ms2_mzs'].apply(lambda x: len(x) if x is not None else 0)
print(f"Average fragment peaks per spectrum: {test_df['n_peaks'].mean():.1f}")

fig, ax = plt.subplots(figsize=(8, 3.5))
test_df['n_peaks'].plot(kind='hist', bins=30, ax=ax, color='#fd9644', edgecolor='black')
ax.set_title("MS2 Fragment Peak Count per Spectrum", fontweight='bold')
ax.set_xlabel("Number of MS2 Peaks")
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 4. The Official Evaluation Metric: MRR@25 on Tautomer InChIKey14

The competition evaluates submissions via Mean Reciprocal Rank on the top 25 candidates:
$$\text{MRR@25} = \frac{1}{M} \sum_{m=1}^{M} \frac{1}{\text{rank}_m} \quad (\text{if rank}_m \le 25, \text{ else } 0)$$

### Key Trap to Avoid:
- The metric normalizes via RDKit `RemoveStereochemistry` and `TautomerEnumerator().Canonicalize()`.
- **10.43% of structures have a tautomer-canonical InChIKey14 differing from their raw InChIKey**.
- Any local evaluation using raw keys understates performance by ~0.007.
- Never submit junk/filler SMILES — unparseable rows get penalized or scored 0.000.
